# B2-023 — Session 1: Adversarial Games

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

A generative adversarial network (GAN) learns to **sample** from a data distribution without ever writing its density down.
Two networks play a game: a generator turns random noise into candidate samples, and a discriminator tries to tell those candidates from real data.
This session builds the tools the game needs and Book 1 never taught: the sigmoid, binary cross-entropy with logits, KL divergence for densities, the Jensen–Shannon divergence, and the pointwise optimization that produces the optimal discriminator.
It ends with the non-saturating generator loss that every practical GAN uses.
Practices after this session: p01, p02, p06, p13.

## 1. Implicit generative models: the generator and the discriminator

A **generative model** produces new samples that look like the training data.
The VAE of [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb) did this with an explicit probabilistic model: a prior, a decoder likelihood, and a lower bound on $\log p(x)$.
A GAN takes a different route and trains a **sampler** directly.

* The **generator** $G_\theta:\mathbb R^k\to\mathbb R^d$ is an MLP (Book 1 `mlp-architecture` with `relu-activation`). It receives a noise vector $z\sim N(0,I_k)$ drawn from a seeded generator and outputs a candidate sample $x=G_\theta(z)$. The distribution of $G_\theta(z)$ is called $p_g$. Nobody writes $p_g$ down: it is **implicit**, defined only by "draw $z$, push it through $G$".
* The **discriminator** $D_\phi:\mathbb R^d\to\mathbb R$ is a second MLP. It outputs one real number per input, a **logit**; Section 2 turns the logit into a probability that the input is real.

The players have opposite goals: $D$ wants to label real rows "real" and generated rows "fake"; $G$ wants its outputs labelled "real".

**This unit's networks and data.** Both networks are tiny MLPs on CPU: the generator is `2→32→32→2` and the discriminator is `2→32→32→1`, with ReLU hidden layers and no activation after the last layer. The target distribution is the unit's seeded `mixture2d` dataset: four Gaussian blobs with standard deviation $0.2$ centred at $(1,1)$, $(-1,1)$, $(-1,-1)$, $(1,-1)$, with 256 training rows and 64 held-out rows. Because the four centres are known, Session 2 can check exactly which blobs a generator reproduces.

**Shape ledger (prose).** A noise batch `z` of shape `(B,2)` gives `G(z)` of shape `(B,2)`; `D(G(z))` has shape `(B,1)`, one logit per row.

**Checkpoint 1A.** For `z` of shape `(64,2)` and the unit's networks, what are the shapes of `G(z)` and `D(G(z))`?

**Checkpoint 1B.** Why can't you evaluate $\log p_g(x)$ for a GAN the way B2-022 evaluated a Gaussian log-density?

**Collected answers.** `(64,2)` and `(64,1)`. Because $p_g$ is implicit: it is the distribution of $G(z)$, and the generator gives a way to sample it, not a formula for its density.

## 2. The sigmoid and binary cross-entropy with logits

Book 1 taught `softmax` and `cross-entropy-loss` for $C$ classes. A discriminator has exactly two classes, real and fake, and this section specializes both tools.

**The sigmoid** (new in this unit) is
$$\sigma(\ell)=\frac{1}{1+e^{-\ell}}.$$
Its properties follow from the formula with Calculus AB:

* $0<\sigma(\ell)<1$, $\sigma(0)=\tfrac12$, $\sigma(\ell)\to1$ as $\ell\to\infty$ and $\to0$ as $\ell\to-\infty$;
* symmetry: $\sigma(-\ell)=\frac{1}{1+e^{\ell}}=\frac{e^{-\ell}}{e^{-\ell}+1}=1-\sigma(\ell)$;
* derivative, by the chain rule: $\sigma'(\ell)=\frac{e^{-\ell}}{(1+e^{-\ell})^2}=\sigma(\ell)\big(1-\sigma(\ell)\big)$.

It is the two-class softmax: the softmax of the logit pair $(\ell,0)$ gives the first class probability $\frac{e^\ell}{e^\ell+e^0}=\sigma(\ell)$.

**Binary cross-entropy (BCE).** For a label $y\in\{0,1\}$ and predicted probability $p=\sigma(\ell)$, Book 1's cross-entropy with two classes becomes
$$\mathrm{BCE}(\ell,y)=-\big[y\log\sigma(\ell)+(1-y)\log(1-\sigma(\ell))\big].$$
With the formulas above, $-\log\sigma(\ell)=\log(1+e^{-\ell})$ and $-\log(1-\sigma(\ell))=\log(1+e^{\ell})$.

**Why "with logits".** Computing $\sigma(\ell)$ first and then taking $\log$ fails for large $|\ell|$: in float32, $\sigma(-100)$ rounds to exactly $0$ and $\log 0=-\infty$. The library function `torch.nn.functional.binary_cross_entropy_with_logits(logits, targets)` takes the **logit** and evaluates $\log(1+e^{\pm\ell})$ in a numerically stable way. Its default reduction is the **mean** over all entries. This unit always passes logits to it, and never applies a sigmoid inside the discriminator.

**Numbers.** For $\ell=2$, $y=1$: $\log(1+e^{-2})=0.126928$. For $\ell=0$ and either label: $\log 2=0.693147$.

**Checkpoint 2A.** Compute $\sigma(0)$ and $\sigma(\ln 3)$ exactly.

**Checkpoint 2B.** Compute the BCE for $\ell=\ln 3$ with label $y=0$, exactly and to six decimals.

**Collected answers.** $\sigma(0)=\tfrac12$; $\sigma(\ln3)=\frac{1}{1+1/3}=\tfrac34$. $-\log(1-\tfrac34)=\log4=1.386294$.

In [ ]:
import torch
import torch.nn.functional as F

logits = torch.tensor([-100.0, -2.0, 0.0, 2.0, 100.0])          # float32
targets = torch.ones_like(logits)                                # label "real"
naive = -torch.log(torch.sigmoid(logits))                        # sigmoid first, then log
stable = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
print("naive :", naive.tolist())
print("stable:", stable.tolist())
print("mean reduction:", F.binary_cross_entropy_with_logits(logits, targets).item())
assert torch.isinf(naive[0]) and torch.isfinite(stable).all()
assert torch.allclose(stable[2], torch.log(torch.tensor(2.0)), atol=1e-6, rtol=1e-6)

## 3. The minimax value function

In theory sections, write $D(x)=\sigma(\text{logit})\in(0,1)$ for the discriminator's probability that $x$ is real. The original GAN objective is the **value function**
$$V(D,G)=E_{x\sim p_{\text{data}}}\big[\log D(x)\big]+E_{z\sim N(0,I)}\big[\log\big(1-D(G(z))\big)\big].$$
The discriminator **maximizes** $V$: it wants $D(x)\to1$ on real rows and $D(G(z))\to0$ on generated rows. The generator **minimizes** $V$: it can only affect the second term, and lowers it by making $D(G(z))$ large. The game is written $\min_G\max_D V(D,G)$.

**From expectations over $z$ to integrals over $x$.** By definition of $p_g$, $E_z[f(G(z))]=E_{x\sim p_g}[f(x)]$ for any function $f$. So
$$V(D,G)=\int\Big[p_{\text{data}}(x)\log D(x)+p_g(x)\log\big(1-D(x)\big)\Big]\,dx.$$

**The batch losses.** Replace each expectation by a batch mean. With Section 2's BCE, $-\frac1B\sum\log D(x_i)$ is the mean BCE of the real logits against label $1$, and $-\frac1B\sum\log(1-D(G(z_i)))$ is the mean BCE of the fake logits against label $0$. Hence the **discriminator loss**
$$\mathcal L_D=\mathrm{BCE}_{\text{mean}}(\text{real logits},1)+\mathrm{BCE}_{\text{mean}}(\text{fake logits},0)=-\hat V,$$
the **sum** of two batch means, and minimizing $\mathcal L_D$ is maximizing $V$.

**The confused discriminator.** If $D\equiv\tfrac12$ (all logits $0$), then $V=\log\tfrac12+\log\tfrac12=-\log4$ and $\mathcal L_D=2\log2=1.386294$. Section 6 shows this is exactly the value at the game's optimum, which is why Session 2 compares training traces with $2\log2$.

**Checkpoint 3A.** A batch has every real logit and every fake logit equal to $0$. What is $\mathcal L_D$?

**Checkpoint 3B.** Which player can change the first term of $V$? Which can change the second?

**Collected answers.** $2\log2=1.386294$, the sum of two means that are each $\log2$. Only $D$ affects the first term; both players affect the second ($D$ through its output, $G$ through the inputs it creates).

## 4. Pointwise optimization and the optimal discriminator

Fix the generator, so $p_g$ is fixed, and ask which discriminator maximizes $V$. By Section 3, $V$ is an integral over $x$ whose integrand at each point depends only on the single number $y=D(x)$:
$$f_x(y)=a\log y+b\log(1-y),\qquad a=p_{\text{data}}(x),\ b=p_g(x).$$
Because $D$ may be any function, its value at one point can be chosen independently of its value at every other point. So the integral is maximized by maximizing the integrand **separately at each $x$**: if at some $x$ the value $D(x)$ did not maximize $f_x$, changing $D$ at that point alone would raise the integrand there and leave it unchanged everywhere else. This is **pointwise optimization** of an integral objective.

**Single-variable calculus.** Assume $a>0$ and $b>0$ at the point. On $y\in(0,1)$,
$$f_x'(y)=\frac ay-\frac{b}{1-y}=0\iff a(1-y)=by\iff y^*=\frac{a}{a+b},$$
and $f_x''(y)=-\frac{a}{y^2}-\frac{b}{(1-y)^2}<0$, so $f_x$ is strictly concave and $y^*$ is its unique maximizer (also $f_x\to-\infty$ at both ends). Therefore
$$D^*(x)=\frac{p_{\text{data}}(x)}{p_{\text{data}}(x)+p_g(x)}.$$

**Boundary convention.** If $a>0=b$, then $f_x(y)=a\log y$ increases on $(0,1)$ and has no interior maximizer; the supremum is approached as $y\to1$, and the convention is $D^*(x)=1$. Symmetrically $a=0<b$ gives $D^*(x)=0$, and points where both densities vanish do not affect $V$. Practices p01, p13, and p14 assume both densities are positive wherever they evaluate $D^*$.

**Worked optimal-discriminator example.** Let $p_{\text{data}}$ be uniform on $[0,2]$, so $p_{\text{data}}(x)=\tfrac12$ there, and let $p_g(x)=1-\tfrac x2$ on $[0,2]$ (a triangle; $\int_0^2(1-\frac x2)dx=2-1=1$).

| $x$ | $p_{\text{data}}$ | $p_g$ | $D^*(x)$ |
|---:|---:|---:|---:|
| $\tfrac12$ | $\tfrac12$ | $\tfrac34$ | $\frac{1/2}{5/4}=\tfrac25$ |
| $1$ | $\tfrac12$ | $\tfrac12$ | $\tfrac12$ |
| $\tfrac32$ | $\tfrac12$ | $\tfrac14$ | $\frac{1/2}{3/4}=\tfrac23$ |

Where the generator over-produces ($x=\tfrac12$), the optimal discriminator leans "fake"; where it under-produces ($x=\tfrac32$), it leans "real"; where the densities agree, it outputs exactly $\tfrac12$. The full proof, with every condition, is p13.

**Checkpoint 4A.** At a point where $p_{\text{data}}(x)=\tfrac13$ and $p_g(x)=\tfrac16$, compute $D^*(x)$.

**Checkpoint 4B.** If $p_g(x)=0<p_{\text{data}}(x)$, why is there no interior maximizer, and what does the convention set?

**Collected answers.** $D^*=\frac{1/3}{1/2}=\tfrac23$. Then $f_x(y)=a\log y$ is strictly increasing on $(0,1)$, so every $y$ is beaten by a larger one; the supremum is approached as $y\to1$, and the convention sets $D^*(x)=1$.

## 5. Continuous KL and the Jensen–Shannon divergence

[B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lessons/02-kl-divergence.ipynb) (Session 2) defined `kl-divergence` for discrete distributions and wrote the density version as an expectation. Written as an integral, for densities $p$ and $q$,
$$\mathrm{KL}(P\|Q)=\int p(x)\log\frac{p(x)}{q(x)}\,dx=E_{x\sim p}\Big[\log\frac{p(x)}{q(x)}\Big],$$
with the same conventions as the discrete case (the integrand is $0$ where $p=0$, and the KL is $+\infty$ if $q=0$ on a region where $p>0$).
**Nonnegativity carries over.** B2-022's argument used only the tangent-line lemma $\log u\le u-1$: with $u=q/p$ on the support of $p$, $-\mathrm{KL}(P\|Q)=\int p\log\frac qp\le\int p\big(\frac qp-1\big)=\int_{p>0}q-1\le0$. So $\mathrm{KL}(P\|Q)\ge0$, with equality exactly when $P=Q$ (this unit states the equality case for densities without the measure-theoretic fine print).

**Jensen–Shannon divergence (new).** Let $M=\tfrac12(P+Q)$, the distribution with density $m=\tfrac12(p+q)$. Then
$$\mathrm{JS}(P\|Q)=\tfrac12\,\mathrm{KL}(P\|M)+\tfrac12\,\mathrm{KL}(Q\|M).$$

* It is **symmetric** in $P$ and $Q$ by construction.
* It is **finite**: wherever $p>0$, $m\ge\frac p2>0$, so $\log\frac pm\le\log2$; hence $0\le\mathrm{JS}\le\log2$.
* It is **nonnegative**, with $\mathrm{JS}(P\|Q)=0$ **iff** $P=Q$: both KL terms are $\ge0$, so the sum is $0$ only if $P=M$ and $Q=M$, which forces $P=Q$; conversely $P=Q$ gives $M=P$ and both terms vanish.

**Discrete numbers.** For $P=(1,0)$ and $Q=(0,1)$ (disjoint supports), $M=(\tfrac12,\tfrac12)$ and each KL is $\log2$, so $\mathrm{JS}=\log2$, the maximum. For $P=(\tfrac12,\tfrac12)$ and $Q=(\tfrac14,\tfrac34)$, $M=(\tfrac38,\tfrac58)$ and the cell below gives $\mathrm{JS}=0.033822$, while $\mathrm{KL}(P\|Q)=0.143841$ and $\mathrm{KL}(Q\|P)=0.130812$ differ.

**Checkpoint 5A.** What is $\mathrm{JS}(P\|Q)$ when $P$ and $Q$ have disjoint supports?

**Checkpoint 5B.** $Q$ is zero on a region where $P$ is positive. Why is $\mathrm{KL}(P\|M)$ still finite?

**Collected answers.** $\log2$: on the support of $P$, $m=p/2$, so $\mathrm{KL}(P\|M)=\int p\log2=\log2$, and likewise for $Q$. Because $m=\frac12(p+q)\ge\frac p2>0$ wherever $p>0$, so the ratio $p/m$ never divides by zero and is at most $2$.

In [ ]:
import math

def kl(p, q):
    return sum(pi * math.log(pi / qi) for pi, qi in zip(p, q) if pi > 0)

def js(p, q):
    m = [(pi + qi) / 2 for pi, qi in zip(p, q)]
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)

P, Q = (0.5, 0.5), (0.25, 0.75)
print(f"JS = {js(P, Q):.6f}  KL(P||Q) = {kl(P, Q):.6f}  KL(Q||P) = {kl(Q, P):.6f}")
print(f"disjoint supports: JS = {js((1.0, 0.0), (0.0, 1.0)):.6f}  log 2 = {math.log(2):.6f}")
assert math.isclose(js(P, Q), js(Q, P), abs_tol=1e-15, rel_tol=0.0)
assert math.isclose(js((1.0, 0.0), (0.0, 1.0)), math.log(2), abs_tol=1e-15, rel_tol=0.0)

## 6. The global optimum of the game

Substitute the optimal discriminator into the value function. The result is the **virtual training criterion** of the generator,
$$C(G)=\max_DV(D,G)=E_{x\sim p_{\text{data}}}\Big[\log\frac{p_{\text{data}}(x)}{p_{\text{data}}(x)+p_g(x)}\Big]+E_{x\sim p_g}\Big[\log\frac{p_g(x)}{p_{\text{data}}(x)+p_g(x)}\Big].$$

**The result (stated here; proved in p14).**
$$C(G)=-\log4+2\,\mathrm{JS}\big(p_{\text{data}}\,\|\,p_g\big).$$
The key idea is that each denominator is twice the mixture density, $p_{\text{data}}+p_g=2m$; p14 asks you to turn that observation into a complete argument, including the conditions under which each step is valid.

**Consequences.** Because $\mathrm{JS}\ge0$ with equality iff the distributions agree (Section 5):

* $C(G)\ge-\log4$ for every generator;
* the minimum $-\log4$ is reached **iff** $p_g=p_{\text{data}}$;
* at that optimum $D^*\equiv\tfrac12$, so $\mathcal L_D=2\log2$ (Section 3).

**What the theory does not promise.** It assumes $D$ is fully optimized for every $G$ and that both networks can represent any function. Real training alternates single gradient steps on finite batches (Session 2), so a discriminator loss near $2\log2$ is consistent with a good generator but does not prove one.

**Numerical check.** For Section 4's uniform/triangle pair, the cell below integrates both sides on $[0,2]$ with a midpoint rule (a Riemann sum, Calculus AB) and finds the same number, $C(G)=-1.278730$ (so $\mathrm{JS}=0.053782$), against $-\log4=-1.386294$.

**Checkpoint 6A.** What is $C(G)$ when $p_g=p_{\text{data}}$?

**Checkpoint 6B.** If $p_g$ and $p_{\text{data}}$ have disjoint supports, what is $C(G)$?

**Collected answers.** $-\log4=-1.386294$. $-\log4+2\log2=0$, the largest possible value, because $\mathrm{JS}=\log2$ for disjoint supports.

In [ ]:
import math

n = 200_000
h = 2.0 / n
xs = [(i + 0.5) * h for i in range(n)]                    # midpoints on [0, 2]
p_data = lambda x: 0.5
p_g = lambda x: 1.0 - x / 2.0

c_of_g = sum(p_data(x) * math.log(p_data(x) / (p_data(x) + p_g(x)))
             + p_g(x) * math.log(p_g(x) / (p_data(x) + p_g(x))) for x in xs) * h
m = lambda x: 0.5 * (p_data(x) + p_g(x))
js = 0.5 * sum(p_data(x) * math.log(p_data(x) / m(x)) + p_g(x) * math.log(p_g(x) / m(x)) for x in xs) * h
print(f"C(G) = {c_of_g:.6f}   -log 4 + 2 JS = {-math.log(4) + 2 * js:.6f}   JS = {js:.6f}")
assert math.isclose(c_of_g, -math.log(4) + 2 * js, abs_tol=1e-9, rel_tol=0.0)

## 7. Worked laboratory: the non-saturating generator loss

**The saturation problem.** In the minimax game the generator minimizes $E_z[\log(1-D(G(z)))]$. Early in training the generator is poor, the discriminator rejects its samples confidently, and the fake logits $\ell$ are very negative. Differentiate with respect to the logit, using $\sigma'=\sigma(1-\sigma)$:
$$\frac{d}{d\ell}\log\big(1-\sigma(\ell)\big)=\frac{-\sigma(\ell)(1-\sigma(\ell))}{1-\sigma(\ell)}=-\sigma(\ell)\ \xrightarrow{\ \ell\to-\infty\ }\ 0.$$
The gradient **vanishes exactly when the generator most needs it**: this loss saturates.

**The non-saturating fix.** Instead, the generator minimizes $-E_z[\log D(G(z))]$, which in code is the mean BCE of the fake logits against label **1**:
$$\frac{d}{d\ell}\big[-\log\sigma(\ell)\big]=-\frac{\sigma(\ell)(1-\sigma(\ell))}{\sigma(\ell)}=-\big(1-\sigma(\ell)\big)\ \xrightarrow{\ \ell\to-\infty\ }\ -1.$$
Both losses push $D(G(z))$ upward, so they share their goal, but the non-saturating loss gives a strong gradient when the generator is losing. The price: it is **not** $-V$ any more, so the identity $C(G)=-\log4+2\,\mathrm{JS}$ describes the minimax game, not the objective the generator actually descends (p14 asks you to say why).

**Worked exam-style example 1 (gradients).** At $\ell=-4$: $\sigma(-4)=0.017986$, so the saturating gradient is $-0.017986$ and the non-saturating gradient is $-0.982014$, about $55$ times larger. The first cell confirms this with autograd for several logits.

**Worked exam-style example 2 (normal form).** At a point with $p_{\text{data}}(x)=\tfrac25$ and $p_g(x)=\tfrac1{10}$, $D^*(x)=\frac{2/5}{2/5+1/10}=\frac{4/10}{5/10}=\tfrac45$. In normal form $a/b$ with $\gcd(a,b)=1$ and $b>0$, $a=4$, $b=5$, and $a+b=9$. Unreduced forms such as $\frac{8}{10}$ (sum $18$) receive no credit in exam register.

**The two losses in code.** The second cell writes both players' losses with BCE-with-logits and checks the reference values at the confused discriminator: $\mathcal L_D=2\log2$ and the non-saturating generator loss $\log2$.

**Checkpoint 7A.** At $\ell=-2$, give both generator gradients with respect to the logit to six decimals.

**Checkpoint 7B.** Do the saturating and non-saturating losses move $D(G(z))$ in the same direction?

**Collected answers.** $\sigma(-2)=0.119203$: saturating $-0.119203$, non-saturating $-(1-0.119203)=-0.880797$. Yes: both decrease as $D(G(z))$ increases; only the size of the gradient differs, dramatically so when $D(G(z))$ is near $0$.

In [ ]:
import torch
import torch.nn.functional as F

for value in (-6.0, -4.0, -2.0, 0.0, 2.0):
    logit = torch.tensor(value, dtype=torch.float64, requires_grad=True)
    saturating = torch.log1p(-torch.sigmoid(logit))            # log(1 - sigma(l)), to be minimized
    (grad_sat,) = torch.autograd.grad(saturating, logit)
    logit2 = torch.tensor(value, dtype=torch.float64, requires_grad=True)
    non_saturating = F.binary_cross_entropy_with_logits(logit2, torch.ones_like(logit2))
    (grad_ns,) = torch.autograd.grad(non_saturating, logit2)
    print(f"logit {value:5.1f}: saturating grad {grad_sat.item():+.6f}   non-saturating grad {grad_ns.item():+.6f}")

In [ ]:
import math
import torch
import torch.nn.functional as F

real_logits = torch.zeros(8, 1)
fake_logits = torch.zeros(8, 1)
loss_d = (F.binary_cross_entropy_with_logits(real_logits, torch.ones_like(real_logits))
          + F.binary_cross_entropy_with_logits(fake_logits, torch.zeros_like(fake_logits)))
loss_g = F.binary_cross_entropy_with_logits(fake_logits, torch.ones_like(fake_logits))
print(f"discriminator loss {loss_d.item():.6f} (2 log 2 = {2 * math.log(2):.6f}); generator loss {loss_g.item():.6f}")
assert math.isclose(loss_d.item(), 2 * math.log(2), abs_tol=1e-6, rel_tol=0.0)

## 8. Common pitfalls and Exam connections

**Common pitfalls (each with its fix).**

1. *Sigmoid, then log.* `torch.log(torch.sigmoid(l))` returns `-inf` for very negative logits and poisons training with `nan`. Fix: `F.binary_cross_entropy_with_logits` on raw logits.
2. *A sigmoid inside the discriminator plus BCE-with-logits.* The sigmoid is then applied twice and the loss can never approach $0$. Fix: the discriminator's last layer is a plain `nn.Linear(32, 1)`.
3. *Swapped labels.* Real rows must have target $1$ and fake rows target $0$ in $\mathcal L_D$. A swapped version trains a discriminator that calls fakes real.
4. *The saturating generator loss.* Minimizing $\log(1-D(G(z)))$ stalls early. Fix: BCE of fake logits against label $1$.
5. *Ratio instead of share.* $D^*=p_{\text{data}}/(p_{\text{data}}+p_g)$, not $p_{\text{data}}/p_g$; the latter is not even a probability.
6. *Halving the discriminator loss.* Averaging the two BCE terms instead of adding them moves the reference value from $2\log2$ to $\log2$. This unit's convention is the sum.
7. *Over-reading the theory.* $C(G)=-\log4+2\,\mathrm{JS}$ assumes an optimal discriminator and the minimax generator loss; it is not a statement about the non-saturating objective or about finite training.

**Exam connections.** Round 2 asks for exact values in normal form (p01: $D^*$ at a point as $a/b$ with $\gcd(a,b)=1$, $b>0$), for the role of each loss (p02), for guarded implementations matched to literal probes (p06: both losses with BCE-with-logits), and for proofs with every condition stated (p13: the optimal discriminator; p14: the JS identity and its equality case). Session 2 turns these losses into the training protocol of p07, p08, p17, and p22.

**Checkpoint 8A.** A teammate's discriminator loss at all-zero logits is $0.693147$. Which convention did they use?

**Checkpoint 8B.** Which practice asks you to prove $C(G)=-\log4+2\,\mathrm{JS}$?

**Collected answers.** They averaged the two BCE terms instead of summing them (pitfall 6). p14, which also asks for the equality case and for why the non-saturating loss is not this quantity.

## 9. Forward-only deeper topic: other divergences and Wasserstein GANs

Forward-only deeper topic.
The JS divergence saturates at $\log2$ when supports are disjoint, which is one explanation of why the minimax generator can receive little useful signal early in training.
Later work replaced it: $f$-GANs train against a whole family of divergences, and Wasserstein GANs train a "critic" whose objective approximates an earth-mover distance between $p_{\text{data}}$ and $p_g$; that distance keeps changing smoothly even when supports are disjoint.
These ideas need optimal-transport duality and a Lipschitz constraint on the critic, neither of which this course teaches.
None of them is a prerequisite, a permitted shortcut, or an assessed API in B2-023.

This final section is forward-only and not assessed.

**Checkpoint 9A.** Is any practice in B2-023 based on a Wasserstein critic?

**Checkpoint 9B.** Which divergence does the original minimax game reduce to at the optimal discriminator?

**Collected answers.** No; it is forward-only and not assessed. The Jensen–Shannon divergence, through $C(G)=-\log4+2\,\mathrm{JS}(p_{\text{data}}\|p_g)$.